# p53 Mutation Dataset – EDA
**Dataset**: p53.fr  |  53,871 rows × 76 columns  |  Target: `Pathogenicity` (5 classes)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_processing.loader import load_raw_csv, basic_info
from src.data_processing.feature_engineering import engineer_features
from config import ACTIVITY_COLS, HOTSPOT_CODONS, LABEL_ORDER
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
PALETTE = ['#2196F3','#4CAF50','#FF9800','#F44336','#9C27B0']

In [ ]:
df = load_raw_csv()
print('Shape:', df.shape)
df.head(3)

## 1. Target: Pathogenicity Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
counts = df['Pathogenicity'].value_counts().reindex(LABEL_ORDER).fillna(0)
bars = ax.bar(counts.index, counts.values, color=PALETTE, edgecolor='white')
ax.bar_label(bars)
ax.set_title('Pathogenicity Class Distribution (53,871 mutations)')
ax.set_ylabel('Count')
plt.xticks(rotation=20, ha='right')
plt.tight_layout(); plt.show()

## 2. Transactivation Activity Scores (WAF1_, MDM2_, BAX_ …)

In [ ]:
act_df = df[ACTIVITY_COLS].apply(pd.to_numeric, errors='coerce')
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, ACTIVITY_COLS):
    act_df[col].hist(bins=40, ax=ax, color='#2196F3', edgecolor='white')
    ax.set_title(col); ax.set_xlabel('Score')
plt.suptitle('Transactivation Activity Score Distributions', fontsize=13)
plt.tight_layout(); plt.show()

## 3. Mutation Codon Distribution with Hotspots

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
df['Codon'].hist(bins=120, ax=ax, color='#2196F3', edgecolor='white')
for c in HOTSPOT_CODONS:
    ax.axvline(c, color='red', linestyle='--', alpha=0.8)
    ax.text(c+1, ax.get_ylim()[1]*0.85, str(c), color='red', fontsize=8)
ax.set_title('Codon Distribution – Red lines = hotspot codons')
ax.set_xlabel('Codon'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()

## 4. Domain vs Pathogenicity

In [ ]:
domain_path = (df.groupby(['Domain','Pathogenicity'])
                 .size().unstack(fill_value=0)
                 .reindex(columns=LABEL_ORDER, fill_value=0))
domain_path = domain_path[domain_path.sum(axis=1) > 100]
domain_path.div(domain_path.sum(axis=1), axis=0).plot(
    kind='bar', stacked=True, figsize=(12, 5), color=PALETTE)
plt.title('Pathogenicity Breakdown by Domain')
plt.ylabel('Fraction'); plt.xlabel('Domain')
plt.xticks(rotation=25, ha='right'); plt.legend(loc='lower right')
plt.tight_layout(); plt.show()

## 5. In-silico Prediction Tools vs Pathogenicity

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['Sift Prediction','Provean_prediction','Polyphen']):
    ct = pd.crosstab(df[col], df['Pathogenicity'])[LABEL_ORDER].fillna(0)
    ct.div(ct.sum(axis=1), axis=0).plot(kind='bar', stacked=True,
        ax=ax, color=PALETTE, legend=False)
    ax.set_title(col); ax.tick_params(axis='x', rotation=20)
axes[2].legend(loc='lower right', fontsize=7)
plt.suptitle('In-silico Tool Predictions vs Pathogenicity')
plt.tight_layout(); plt.show()

## 6. Activity Score Boxplot by Pathogenicity

In [ ]:
df_plot = df[ACTIVITY_COLS + ['Pathogenicity']].copy()
for c in ACTIVITY_COLS:
    df_plot[c] = pd.to_numeric(df_plot[c], errors='coerce')
df_plot['mutation_activity_mean'] = df_plot[ACTIVITY_COLS].mean(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
order = LABEL_ORDER
df_plot.boxplot(column='mutation_activity_mean', by='Pathogenicity',
                ax=ax, order=order)
ax.set_title('Mean Activity Score by Pathogenicity Class')
ax.set_xlabel('Pathogenicity'); ax.set_ylabel('Mean Activity Score')
plt.suptitle('')
plt.xticks(rotation=20, ha='right')
plt.tight_layout(); plt.show()

## 7. PCA Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for i, (label, color) in enumerate(zip(LABEL_ORDER, PALETTE)):
    subset = df[df['Pathogenicity'] == label]['PCA_Score'].dropna()
    subset.hist(bins=40, ax=ax, alpha=0.6, color=color, label=label)
ax.set_title('PCA_Score Distribution by Pathogenicity')
ax.set_xlabel('PCA Score'); ax.set_ylabel('Count')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 8. Engineered Features Preview

In [ ]:
df_eng  = engineer_features(df)
new_cols = [c for c in df_eng.columns if c not in df.columns]
print('New engineered features:', new_cols)
df_eng[new_cols].describe()